In [8]:
#IMPORTAMOS LAS LIBRERÍAS NECESARIAS
from os import listdir
from numpy import asarray
from numpy import save
import tensorflow as tf
#tf.config.set_visible_devices([], 'GPU')
from tensorflow.keras.utils import load_img
from tensorflow.keras.utils import img_to_array
import pandas as pd
from sklearn.model_selection import train_test_split
from tensorflow import keras
from tensorflow.keras.layers import Conv2D
from tensorflow.keras.layers import MaxPool2D
from tensorflow.keras.layers import Flatten


In [9]:
import tensorflow as tf

# Lista las GPUs disponibles
gpus = tf.config.list_physical_devices('GPU')
print("GPUs detectadas:", gpus)

# Info más detallada
print("Versión de TensorFlow:", tf.__version__)
print("CUDA disponible:", tf.test.is_built_with_cuda())
print("GPU disponible:", tf.test.is_gpu_available())  # deprecated pero útil

GPUs detectadas: []
Versión de TensorFlow: 2.21.0
CUDA disponible: True
GPU disponible: False


W0000 00:00:1777569848.037558   32499 gpu_device.cc:2365] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


In [10]:
#CARGAMOS EL DATASET QUE RELACIONA LAS FOTOS CON SUS CARACTERÍSTICAS. NUESTRO SISTEMA TIENE QUE APRENDER A PREDECIR ESTOS ATRIBUTOS EN BASE A LAS FOTOS RECIBIDAS.
#SI LA PERSONA ES CALVA O NO, SI ESTÁ SONRIENDO O SI TIENE EL PELO LISO SON CARACTERÍSTICAS DE LAS PERSONAS DE LAS FOTOS QUE EL SISTEMA TIENE QUE APRENDER A 
#PREDECIR.
df = pd.read_csv("list_attr_celeba.csv")
df.replace(-1,0,inplace=True)
df.shape

(202599, 41)

In [11]:
import pandas as pd
df_seleccionado = pd.DataFrame()
gafas = []
sonriendo = []
image_id = []
photos = []
singafas = 0
for idx,row in df.iterrows():
    if row['Eyeglasses']:
        gafas.append(row['Eyeglasses'])
        sonriendo.append(row['Smiling'])
        image_id.append(row['image_id'])
        photo = load_img('img_align_celeba/img_align_celeba/' + row['image_id'], target_size=(50,50), color_mode='grayscale')
        photos.append(img_to_array(photo)/255.)
        del photo
    else:
        if singafas < 13193:
            gafas.append(row['Eyeglasses'])
            sonriendo.append(row['Smiling'])
            image_id.append(row['image_id'])
            singafas += 1
            photo = load_img('img_align_celeba/img_align_celeba/' + row['image_id'], target_size=(50,50), color_mode='grayscale')
            photos.append(img_to_array(photo)/255.)
            del photo
df_seleccionado['Eyeglasses'] = gafas
df_seleccionado['Smiling'] = sonriendo
df_seleccionado['image_id'] = image_id
photos = asarray(photos)
#13193

In [12]:
photos.shape

(26386, 50, 50, 1)

In [13]:
#VAMOS A HACER UN PRIMER INTENTO CON UNA RED NEURONAL NORMAL. COMO SABÉIS NECESITA UNA ENTRADA EN DOS DIMENSIONES. 
#POR ESO METEMOS CAPA FLATTEN.
#HACEMOS UNA PRUEBA COGIENDO SOLO 3 ATRIBUTOS (QUE, EN PRINCIPIO NO TIENEN QUE VER CON EL COLOR DE LAS IMÁGENES)
#HAY QUE ACORDARSE DE LIMITAR LA Y PARA COGER SOLO 100000 FILAS COMO HICIMOS CUANDO COGIMOS LAS FOTOS.
X_train, X_test, y_train, y_test = train_test_split(photos, df_seleccionado[['Eyeglasses','Smiling']], test_size = 0.1, random_state = 0)

In [14]:
#CREAMOS UNA RED NEURONAL NORMAL PARA VER QUE TAL FUNCIONA CON ESTE DATASET. A PRIORI PODRÍA FUNCIONAR BIEN, YA QUE LAS FOTOS ESTÁN BASTANTE CENTRADAS.
model = keras.models.Sequential()
model.add(keras.layers.Flatten(input_shape=[50, 50, 1]))
model.add(keras.layers.Dense(500,activation='relu',kernel_initializer='he_normal'))
model.add(keras.layers.BatchNormalization())
model.add(keras.layers.Dense(100,activation='relu',kernel_initializer='he_normal'))
model.add(keras.layers.BatchNormalization())
model.add(keras.layers.Dense(2,activation='sigmoid',kernel_initializer='glorot_normal'))

/home/ciabd01/anaconda3/lib/python3.13/site-packages/keras/src/layers/reshaping/flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [15]:
#VAMOS A ENTREARLA UN POCO (5 ÉPOCAS)
model.compile(loss='binary_crossentropy', optimizer = keras.optimizers.Adam(learning_rate=0.001, beta_1=0.9, beta_2=0.999), metrics=['binary_accuracy'])
early_stopping_cb = keras.callbacks.EarlyStopping(patience=5,
restore_best_weights=True)
history = model.fit(X_train, y_train, epochs=5,validation_split = 0.1,callbacks=[early_stopping_cb])

Epoch 1/5
668/668 ━━━━━━━━━━━━━━━━━━━━ 6s 7ms/step - binary_accuracy: 0.8031 - loss: 0.4226 - val_binary_accuracy: 0.7495 - val_loss: 0.5499
Epoch 2/5
668/668 ━━━━━━━━━━━━━━━━━━━━ 5s 7ms/step - binary_accuracy: 0.8456 - loss: 0.3517 - val_binary_accuracy: 0.8349 - val_loss: 0.3761
Epoch 3/5
668/668 ━━━━━━━━━━━━━━━━━━━━ 5s 7ms/step - binary_accuracy: 0.8523 - loss: 0.3355 - val_binary_accuracy: 0.8581 - val_loss: 0.3242
Epoch 4/5
668/668 ━━━━━━━━━━━━━━━━━━━━ 5s 7ms/step - binary_accuracy: 0.8580 - loss: 0.3223 - val_binary_accuracy: 0.8242 - val_loss: 0.3891
Epoch 5/5
668/668 ━━━━━━━━━━━━━━━━━━━━ 5s 7ms/step - binary_accuracy: 0.8643 - loss: 0.3110 - val_binary_accuracy: 0.7792 - val_loss: 0.5132


In [16]:
history = model.fit(X_train, y_train, epochs=5,validation_split = 0.1,callbacks=[early_stopping_cb])

Epoch 1/5
668/668 ━━━━━━━━━━━━━━━━━━━━ 5s 7ms/step - binary_accuracy: 0.8604 - loss: 0.3226 - val_binary_accuracy: 0.8408 - val_loss: 0.3663
Epoch 2/5
668/668 ━━━━━━━━━━━━━━━━━━━━ 5s 7ms/step - binary_accuracy: 0.8633 - loss: 0.3124 - val_binary_accuracy: 0.8204 - val_loss: 0.4041
Epoch 3/5
668/668 ━━━━━━━━━━━━━━━━━━━━ 5s 7ms/step - binary_accuracy: 0.8699 - loss: 0.3014 - val_binary_accuracy: 0.8672 - val_loss: 0.3169
Epoch 4/5
668/668 ━━━━━━━━━━━━━━━━━━━━ 5s 7ms/step - binary_accuracy: 0.8728 - loss: 0.2940 - val_binary_accuracy: 0.8615 - val_loss: 0.3342
Epoch 5/5
668/668 ━━━━━━━━━━━━━━━━━━━━ 5s 7ms/step - binary_accuracy: 0.8777 - loss: 0.2835 - val_binary_accuracy: 0.8543 - val_loss: 0.3653


In [23]:
import numpy as np
y_pred = (model.predict(X_test) > 0.5).astype(int)
print(accuracy_score(y_test['Eyeglasses'],y_pred[:,0]))

83/83 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step  
0.8548692686623721


In [24]:
model.evaluate(X_test,y_test)


83/83 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - binary_accuracy: 0.8602 - loss: 0.3250


[0.3250200152397156, 0.860174298286438]

In [ ]:
#VAMOS A PROBAR AHORA CON UNA CONVOLUCIONAL. 

In [ ]:
model = keras.models.Sequential([
    # Bloque 1
    Conv2D(32, (3,3), activation='relu', padding='same', input_shape=(50,50,1)),
    Conv2D(32, (3,3), activation='relu', padding='same'),
    MaxPool2D(2,2),
    keras.layers.BatchNormalization(),
    keras.layers.Dropout(0.25),

    # Bloque 2
    Conv2D(64, (3,3), activation='relu', padding='same'),
    Conv2D(64, (3,3), activation='relu', padding='same'),
    MaxPool2D(2,2),
    keras.layers.BatchNormalization(),
    keras.layers.Dropout(0.25),

    # # Bloque 3
    # Conv2D(128, (3,3), activation='relu', padding='same'),
    # MaxPool2D(2,2),
    # keras.layers.BatchNormalization(),
    # keras.layers.Dropout(0.25),

    Flatten(),

    # Cabeza densa — más capacidad que antes
    keras.layers.Dense(128, activation='relu', kernel_initializer='he_normal'),
    keras.layers.Dropout(0.5),
    keras.layers.Dense(64, activation='relu', kernel_initializer='he_normal'),

    keras.layers.Dense(2, activation='sigmoid')
])

In [ ]:
model.compile(loss='binary_crossentropy', optimizer = keras.optimizers.Adam(learning_rate=0.001, beta_1=0.9, beta_2=0.999), metrics=['accuracy'])
early_stopping_cb = keras.callbacks.EarlyStopping(patience=10,
restore_best_weights=True)
history = model.fit(X_train, y_train, epochs=1000,validation_split = 0.2,callbacks=[early_stopping_cb])

Epoch 1/1000
594/594 ━━━━━━━━━━━━━━━━━━━━ 63s 101ms/step - accuracy: 0.7184 - loss: 0.4451 - val_accuracy: 0.7560 - val_loss: 0.2774
Epoch 2/1000
594/594 ━━━━━━━━━━━━━━━━━━━━ 59s 100ms/step - accuracy: 0.7591 - loss: 0.2641 - val_accuracy: 0.7154 - val_loss: 0.2343
Epoch 3/1000
594/594 ━━━━━━━━━━━━━━━━━━━━ 56s 95ms/step - accuracy: 0.7681 - loss: 0.2191 - val_accuracy: 0.7733 - val_loss: 0.1928
Epoch 4/1000
594/594 ━━━━━━━━━━━━━━━━━━━━ 54s 92ms/step - accuracy: 0.7681 - loss: 0.1982 - val_accuracy: 0.7758 - val_loss: 0.2629
Epoch 5/1000
594/594 ━━━━━━━━━━━━━━━━━━━━ 55s 93ms/step - accuracy: 0.7711 - loss: 0.1837 - val_accuracy: 0.8623 - val_loss: 0.1992
Epoch 6/1000
594/594 ━━━━━━━━━━━━━━━━━━━━ 55s 92ms/step - accuracy: 0.7704 - loss: 0.1735 - val_accuracy: 0.7604 - val_loss: 0.1832
Epoch 7/1000
594/594 ━━━━━━━━━━━━━━━━━━━━ 56s 94ms/step - accuracy: 0.7758 - loss: 0.1601 - val_accuracy: 0.8337 - val_loss: 0.1821
Epoch 8/1000
594/594 ━━━━━━━━━━━━━━━━━━━━ 56s 94ms/step - accuracy: 0.7776

In [18]:
from sklearn.metrics import accuracy_score

In [25]:
y_pred = (model.predict(X_test) > 0.5).astype(int)
print(accuracy_score(y_test['Smiling'],y_pred[:,1]))

83/83 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step  
0.8654793482379689


In [ ]:
y_pred = model.predict(X_test)

83/83 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step


In [ ]:
print(y_pred[0:10])

[[0. 0.]
 [0. 0.]
 [0. 0.]
 [0. 0.]
 [0. 0.]
 [0. 0.]
 [0. 0.]
 [0. 0.]
 [0. 0.]
 [0. 0.]]


Este fenómeno es el famoso gradient explosion. 
El modelo empieza a aprender correctamente
Los gradientes se acumulan y se vuelven enormes
Los pesos se actualizan con valores gigantes.
Posibles soluciones:
- Bajar el LR.
- Meter en el optimizador la opción clipnorm=1.0
- Normalizar los datos de entrada (ya lo hemos hecho)
- Meter capas de batchnormalization.

Bajando el LR ya no pasa. Pero no mejora el resultado de validación porque pone todo 0's.


In [ ]:
model.evaluate(X_test,y_test)
#SERÍA BUENO PROBAR CON UNA RED MÁS GRANDE TANTO EN CAPAS CONVOLUCIONALES COMO EN NEURONAS DE LA PARTE FULLY CONNECTED.
#SEGURAMENTE OBTENDRÍAMOS UN RESULTADO BASTANTE MEJOR.

83/83 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.6825 - loss: 0.5295


[0.5295066237449646, 0.6824554800987244]

In [ ]:
y_pred = model.predict(X_test)
for prediccion in y_pred:
    print(prediccion.round())

32/32 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step
[0. 0. 0.]
[0. 0. 0.]
[0. 0. 0.]
[0. 0. 0.]
[0. 0. 0.]
[0. 0. 0.]
[0. 0. 0.]
[0. 0. 0.]
[0. 0. 0.]
[0. 0. 0.]
[0. 0. 0.]
[0. 0. 0.]
[0. 0. 0.]
[0. 0. 0.]
[0. 0. 0.]
[0. 0. 0.]
[0. 0. 0.]
[0. 0. 0.]
[0. 0. 0.]
[0. 0. 0.]
[0. 0. 0.]
[0. 0. 0.]
[0. 0. 0.]
[0. 0. 0.]
[0. 0. 0.]
[0. 0. 0.]
[0. 0. 0.]
[0. 0. 0.]
[0. 0. 0.]
[0. 0. 0.]
[0. 0. 0.]
[0. 0. 0.]
[0. 0. 0.]
[0. 0. 0.]
[0. 0. 0.]
[0. 0. 0.]
[0. 0. 0.]
[0. 0. 0.]
[0. 0. 0.]
[0. 0. 0.]
[0. 0. 0.]
[0. 0. 0.]
[0. 0. 0.]
[0. 0. 0.]
[0. 0. 0.]
[0. 0. 0.]
[0. 0. 0.]
[0. 0. 0.]
[0. 0. 0.]
[0. 0. 0.]
[0. 0. 0.]
[0. 0. 0.]
[0. 0. 0.]
[0. 0. 0.]
[0. 0. 0.]
[0. 0. 0.]
[0. 0. 0.]
[0. 0. 0.]
[0. 0. 0.]
[0. 0. 0.]
[0. 0. 0.]
[0. 0. 0.]
[0. 0. 0.]
[0. 0. 0.]
[0. 0. 0.]
[0. 0. 0.]
[0. 0. 0.]
[0. 0. 0.]
[0. 0. 0.]
[0. 0. 0.]
[0. 0. 0.]
[0. 0. 0.]
[0. 0. 0.]
[0. 0. 0.]
[0. 0. 0.]
[0. 0. 0.]
[0. 0. 0.]
[0. 0. 0.]
[0. 0. 0.]
[0. 0. 0.]
[0. 0. 0.]
[0. 0. 0.]
[0. 0. 0.]
[0. 0. 0.]
[0. 0. 0.]
[0. 0. 0.]
[0. 0. 0.]
[0.